# 광각(top) 검출모델 학습 — wide_v1 (깡통 yolov8n)

광각캠은 **어안·top-down**이라 본체 `cube.pt` 도메인 밖 → **광각 전용 데이터로 새 모델** 학습.

- 업로드: `wide_dataset_2026-06-30.zip` (cube_v3 pre-label → 노트북 labelImg 교정본). 내부 루트 `wide_dataset/`.
- **본체 cube.pt와 별개 모델.** 결과 best.pt는 젯슨에서 `models/wide.pt`로 배포(광각캠 검출용). cube.pt는 본체용 유지.
- 런타임이 **raw 어안 프레임(180° flip)**에서 검출 → 학습도 그 프레임 그대로(rectify 안 함). 왜곡보정은 투영단계(world_model)에서.
- **imgsz=1280** — 광각은 물체가 작고 멀어 1280 필요(런타임도 1280). 런타임=학습 일치.
- 런타임 → GPU 켜고 실행.

In [ ]:
# [셀1] 업로드 + 압축해제 (기존 /content/wide_dataset 제거 후 — 라운드 간 섞임 방지)
from google.colab import files
import os, shutil
up = files.upload()                # wide_dataset_*.zip 선택
ZIP = next(iter(up))
shutil.rmtree('/content/wide_dataset', ignore_errors=True)
!unzip -o -q "$ZIP" -d /content
print('uploaded:', ZIP, '-> extracted:', sorted(os.listdir('/content/wide_dataset')))

In [ ]:
# [셀2] train/val 분리 + 코랩용 data.yaml 생성
import os, glob, random, shutil, yaml
random.seed(0)
ROOT = '/content/wide_dataset'
HELD_OUT_VAL = True      # ← False면 train==val (87장 작으니 전체학습 원하면)
VAL_FRAC = 0.15
lbl = lambda p: f"{ROOT}/labels/" + os.path.splitext(os.path.basename(p))[0] + ".txt"
pairs = [(i, lbl(i)) for i in sorted(glob.glob(f'{ROOT}/images/*')) if os.path.exists(lbl(i))]
print('pairs:', len(pairs), ' (빈 라벨=배경음성 포함)')
if HELD_OUT_VAL:
    random.shuffle(pairs); n = int(len(pairs) * VAL_FRAC)
    for split, items in [('train', pairs[n:]), ('val', pairs[:n])]:
        for s in ('images', 'labels'):
            os.makedirs(f'{ROOT}/{split}/{s}', exist_ok=True)
        for img, lb in items:
            shutil.copy(img, f'{ROOT}/{split}/images/'); shutil.copy(lb, f'{ROOT}/{split}/labels/')
    print('train', len(pairs) - n, '/ val', n)
    data = dict(path=ROOT, train='train/images', val='val/images')
else:
    data = dict(path=ROOT, train='images', val='images')
data.update(nc=4, names=['cube', 'octahedron', 'dodecahedron', 'icosahedron'])
yaml.safe_dump(data, open(f'{ROOT}/data_colab.yaml', 'w'))
print(open(f'{ROOT}/data_colab.yaml').read())

In [ ]:
# [셀3] 학습 — 광각 전용, 깡통 yolov8n(사전학습 백본)에서 새로. imgsz=1280(런타임과 일치)
!pip -q install ultralytics
from ultralytics import YOLO
YOLO('yolov8n.pt').train(data='/content/wide_dataset/data_colab.yaml',
    epochs=100, imgsz=1280, batch=8, patience=30, name='wide_v1')

In [ ]:
# [셀4] best.pt 내려받기 → 젯슨 models/wide.pt 로 배포(광각캠 검출용, cube.pt는 본체용 유지)
from google.colab import files
files.download('runs/detect/wide_v1/weights/best.pt')